# **MAIN FUNCTION**
# **WEBSTIE FOR FAKE NEWS DETECTION**

In [ ]:
!pip install pyngrok

In [ ]:
!pip install pyngrok flask groq PyPDF2 nltk joblib scikit-learn

In [ ]:
import os
import re
import numpy as np
import joblib
import nltk
from nltk.corpus import stopwords
from flask import Flask, request, jsonify, render_template
from pyngrok import ngrok
from PyPDF2 import PdfReader
from groq import Groq

In [ ]:
!pip install groq

In [ ]:
# --- One-time NLTK setup ---
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords")

STOP_WORDS = set(stopwords.words("english"))

IMPORT ML_MODELS.PKL TO USE THE MODELS

In [ ]:
bundle = joblib.load("ml_models.pkl")

LR = bundle["LR"]
DTC = bundle["DTC"]
RFC = bundle["RFC"]
Gbc = bundle["GBC"]

vectorization = bundle["vectorizer"]
model_weights = bundle["weights"]

In [ ]:
from groq import Groq

API_KEY = "YOur_api"

client = Groq(api_key=API_KEY)

def groq_fake_news(text):
    """
    Classify news text as Real or Fake using Groq.
    Returns:
        label (Real/Fake)
        reason (short explanation)
    """

    prompt =  f"""
You are an expert fact-checking assistant.

The text below may be a full news article, OR a short claim, tweet, or headline.
Do NOT judge it as fake simply because it is brief, lacks a byline, lacks narrative
structure, or doesn't look like a formal article. Short claims are still valid input —
judge them the same way a fact-checker would judge a tweet: based on the plausibility
and factual accuracy of the claim itself, not its format or length.

Evaluate:
1. Factual accuracy and plausibility of the claim
2. Internal logical consistency
3. Whether the claim matches known, verifiable facts
4. Sensational or misleading language (if present)
5. Signs of fabrication or misinformation

Do NOT evaluate: article length, presence of sources/citations, narrative structure,
or whether it "reads like journalism." A short true claim is still Real. A short false
claim is still Fake.

Respond ONLY in this exact format:

Label: Real/Fake
Confidence: <0-100>
Reason: <one short sentence>

Text to evaluate:
{text}
"""

    try:

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        ai_text = response.choices[0].message.content.strip()

        label = None
        reason = None
        confidence = None

        for line in ai_text.split("\n"):

            if line.lower().startswith("label:"):
                label = line.split(":", 1)[1].strip()

            elif line.lower().startswith("confidence:"):
                    value = line.split(":", 1)[1].strip().replace("%", "")

                    try:
                        confidence = float(value)
                    except ValueError:
                        confidence = None
            elif line.lower().startswith("reason:"):
                reason = line.split(":", 1)[1].strip()


        if label is None:
            label = "Unknown"

        if reason is None:
            reason = "No explanation provided."

        return label, confidence, reason

    except Exception as e:
        return "Unknown", str(e)

In [ ]:
def cleaner(text):

  stop_words = set(stopwords.words('english'))
  #convert to lower case
  text = text.lower()
  #remove URLS
  text = re.sub(r'https?://\S+|www\.\S+', '', text)
  #remove html tags
  text = re.sub(r'<.*?>', '', text)
  #remove punctuations
  text = text = re.sub(r'[^\w\s]', '', text)
  #remove digits
  text = re.sub(r'\d', '', text)
  # remove emojis & special characters
  text = re.sub(r'[^\x00-\x7F]+', '', text)
  # remove newlines & extra spaces
  text = re.sub(r'\s+', ' ', text).strip()
  # remove stopwords
  text = ' '.join(word for word in text.split() if word not in stop_words)

  return text


In [ ]:

def weighted_ml_verdict(news_content):
    """Combine per-model probabilities into a single weighted 0-1 score."""
    cleaned = cleaner(news_content)
    vec = vectorization.transform([cleaned])

    weighted_sum = 0.0
    total_weight = 0.0
    for name, model in MODELS.items():
        prob_real = model.predict_proba(vec)[0][1]
        w = model_weights[name]
        weighted_sum += prob_real * w
        total_weight += w

    # Normalize in case weights don't already sum to 1
    return weighted_sum / total_weight if total_weight else weighted_sum


In [ ]:
def evaluate_and_output(news_content, low=0.35, high=0.75):
    """
    Returns a tuple: (result, confidence, reason)
      result: 1 (True/Real) or 0 (Fake)
    """
    score = weighted_ml_verdict(news_content)

    confidence = score * 100 if score >= 0.5 else (1 - score) * 100

    print(f"Weighted ML Score : {score:.3f}")
    print(f"Confidence        : {confidence:.2f}%")

    if score > high:
        print("Verdict           : True News (ML Confident)")
        return 1, confidence, "Classified confidently by the ML ensemble."

    elif score <= low:
        print("Verdict           : Fake News (ML Confident)")
        return 0, confidence, "Classified confidently by the ML ensemble."

    else:
        print("ML Confidence Low - Consulting AI...")
        label, ai_confidence, reason = groq_fake_news(news_content)

        if label.lower() == "real":
            verdict = "True News"
            result = 1
        else:
            verdict = "Fake News"
            result = 0

        final_confidence = ai_confidence if ai_confidence is not None else confidence

        print(f"Verdict           : {verdict} (AI Assisted)")
        print(f"AI Reason         : {reason}")
        return result, final_confidence, reason



In [ ]:
def final_decision(news_content):
    """Wrapper used by the Flask route: returns (result, confidence, reason)."""
    return evaluate_and_output(news_content)

In [ ]:

MODELS = {
    "Logistic Regression": LR,
    "Decision Tree": DTC,
    "Random Forest": RFC,
    "Gradient Boosting": Gbc,
}


# --- File reading ---
def read_pdf(file_path):
    text = ""
    reader = PdfReader(file_path)
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted
    return text


def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()



In [ ]:
# Set your ngrok authtoken (free, from https://dashboard.ngrok.com/get-started/your-authtoken)
# Do NOT leave a real token hardcoded in a notebook you plan to share.
ngrok.set_auth_token(os.environ.get("NGROK_AUTHTOKEN", "your_id"))

In [ ]:
# --- Make sure Flask can find index.html ---
# render_template() only looks inside a folder literally named "templates".
# If index.html is sitting in the Colab root instead, this copies it in.
import shutil

os.makedirs("templates", exist_ok=True)
if os.path.exists("index.html") and not os.path.exists("templates/index.html"):
    shutil.copy("index.html", "templates/index.html")
    print("Copied index.html into templates/")
elif os.path.exists("templates/index.html"):
    print("templates/index.html already present.")
else:
    print("WARNING: index.html not found in the Colab root or templates/ folder.")

templates/index.html already present.


In [ ]:
from pyngrok import ngrok

# --- Flask app ---
app = Flask(__name__)


@app.route("/", methods=["GET"])
def home():
    return render_template("index.html")


@app.route("/analyze", methods=["POST"])
def analyze():
    news_text = request.form.get("news_text")
    news_file = request.files.get("news_file")

    try:
        if news_file and news_file.filename != "":
            filename = news_file.filename.lower()
            if filename.endswith(".pdf"):
                news_text = read_pdf(news_file)
            else:
                news_text = news_file.read().decode("utf-8", errors="ignore")

        if not news_text or news_text.strip() == "":
            return jsonify({"error": "No content provided"}), 400

        result, confidence, reason = final_decision(news_text)

        return jsonify({
            "result": "Real" if result == 1 else "Fake",
            "reason": reason,
            "confidence": round(confidence, 2) if confidence is not None else None,
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500


def run_app():
    app.run(port=5000, debug=False, use_reloader=False)


# --- Colab-safe launch ---
# Kill any previously open tunnel (free ngrok accounts allow only 1 at a time;
# re-running this cell without killing the old tunnel causes drops/failures).
ngrok.kill()

public_url = ngrok.connect(5000)
print("Public URL:", public_url)

# Run Flask in a background thread so this cell doesn't block Colab,
# which is what was causing the tunnel to die before.
import threading
thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

# Keep the main cell alive so Colab doesn't tear down the process.
import time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.disconnect(public_url.public_url)
    ngrok.kill()

Public URL: NgrokTunnel: "https://unacademically-subcheliform-alease.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 04:17:18] "GET / HTTP/1.1" 200 -


Weighted ML Score : 0.573
Confidence        : 57.35%
ML Confidence Low - Consulting AI...


INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 04:17:59] "POST /analyze HTTP/1.1" 200 -


Verdict           : Fake News (AI Assisted)
AI Reason         : Portugal have never lost 4-0 to New Zealand in the FIFA World Cup, and the described events do not match historical records.
Weighted ML Score : 0.561
Confidence        : 56.06%
ML Confidence Low - Consulting AI...


...

INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 04:19:07] "POST /analyze HTTP/1.1" 200 -


Weighted ML Score : 0.811
Confidence        : 81.07%
Verdict           : True News (ML Confident)


ERROR:pyngrok.process.ngrok:t=2026-07-08T04:19:31+0000 lvl=eror msg="internal error" pg=/api/tunnels/http-5000-2a06a6a7-7fe8-48e1-83ac-cdbdfab679cb id=bfbfe7cf0fbddfca err="session closed"


Shutting down...


PyngrokNgrokHTTPError: ngrok client exception, API returned 500: {"error_code":107,"status_code":500,"msg":"internal error","details":{"err":"session closed"}}
